## config

In [ ]:
import os
import sys
from pathlib import Path

ROOT = "/home/wangxc1117/STDK_GNA_Research"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
import optuna

torch.set_default_dtype(torch.float32)
optuna.logging.set_verbosity(optuna.logging.WARNING)

from examples.baselines.stdk.st_interp import create_model
from spatial_adapter.models.spatial_adapter import (
    SpatialAdapterConfig,
    ADMMConfig,
    TrainingConfig,
    BasisConfig,
)
from examples.baselines.timesplit.experiment_core import (
    seed_everything,
    build_fixed_location_subset,
    build_contiguous_time_splits,
    build_stdk_model_config,
    train_simple_loop,
    predict_all_simple,
    rmse_pooled,
    rmse_on_time_subset,
    cov_frob_from_field,
    DictDataset,
    collate_fn,
    new_trend_basis,
    fit_adapter_reconstruct_all_times,
    plot_trial_maps,
    fmt_pm,
    build_standard_bin_edges,
    sv_match_loss_for_prediction_matrix,
)

SEED = 123
DATA_DIR = Path("/home/wangxc1117/study-DeepKriging/Space-Time.DeepKriging/simulation_2b-8/data")
FULL_FILE = "2b_8.csv"

EPOCHS = 350
BATCH_SIZE = 512
LR = 1e-3
WEIGHT_DECAY = 1e-5

SPACE_RATIO_KEEP = 0.1

TRAIN_RATIO_TIME = 0.1
VAL_RATIO_TIME = 0.1
TEST_RATIO_TIME = 0.8

GNA_BATCH_SIZE = 64
N_TRIALS = 50
TAU_MIN = 1e-8
TAU_MAX = 1e4

K_FIXED = 80
N_RUNS_FIXED_K = 100

SEMIVAR_WEIGHTED = True
SEMIVAR_NORMALIZED = False
SEMIVAR_ESTIMATOR = "matheron"

TUNING_TARGET = "covfrob"

if TUNING_TARGET not in {"rmse", "covfrob", "sv_score"}:
    raise ValueError("TUNING_TARGET must be one of {'rmse', 'covfrob', 'sv_score'}")

RESULT_DIR = Path("./2b_8")
REPEAT_DIR = RESULT_DIR / (
    f"{TUNING_TARGET}_tuning"
    f"/k_{K_FIXED}_fixedspace{SPACE_RATIO_KEEP}_time_train{TRAIN_RATIO_TIME}_val{VAL_RATIO_TIME}_test{TEST_RATIO_TIME}"
)
REPEAT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_CSV = REPEAT_DIR / f"k_{K_FIXED}_repeat_runs_summary.csv"

PRED_DIR = REPEAT_DIR / "saved_predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

TRIAL_DIR = REPEAT_DIR / "trial_results"
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

PHI_DIR = REPEAT_DIR / "saved_phi"
PHI_DIR.mkdir(parents=True, exist_ok=True)

ALL_TRIAL_CSV = TRIAL_DIR / "all_trial_results.csv"

DIAG_DIR = REPEAT_DIR / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device, flush=True)

seed_everything(SEED)

config = SpatialAdapterConfig(
    admm=ADMMConfig(
        rho=1.0,
        dual_momentum=0.2,
        max_iters=3000,
        min_outer=20,
        tol=1e-4,
    ),
    training=TrainingConfig(
        lr_mu=1e-2,
        batch_size=GNA_BATCH_SIZE,
        pretrain_epochs=5,
    ),
    basis=BasisConfig(
        phi_every=5,
        phi_freeze=200,
    ),
)

def choose_objective_value(val_rmse, covfrob_reg_val, sv_loss_val):
    if TUNING_TARGET == "rmse":
        return float(val_rmse)
    if TUNING_TARGET == "covfrob":
        return float(covfrob_reg_val)
    if TUNING_TARGET == "sv_score":
        return float(sv_loss_val)
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")

def objective_label():
    if TUNING_TARGET == "rmse":
        return "best_val_rmse"
    if TUNING_TARGET == "covfrob":
        return "best_val_covfrob"
    if TUNING_TARGET == "sv_score":
        return "best_val_sv_score"
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")

def objective_best_by_column():
    if TUNING_TARGET == "rmse":
        return "val_rmse"
    if TUNING_TARGET == "covfrob":
        return "covfrob_reg_val"
    if TUNING_TARGET == "sv_score":
        return "sv_loss_val"
    raise ValueError(f"Unknown TUNING_TARGET: {TUNING_TARGET}")

def load_all_trials_or_empty():
    if ALL_TRIAL_CSV.exists():
        return pd.read_csv(ALL_TRIAL_CSV)
    return pd.DataFrame()

def load_summary_or_empty():
    if SUMMARY_CSV.exists():
        return pd.read_csv(SUMMARY_CSV)
    return pd.DataFrame()

def list_prediction_npz_files():
    return sorted(PRED_DIR.glob("seed_*.npz"))

def print_main_paths():
    print("REPEAT_DIR:", REPEAT_DIR, flush=True)
    print("SUMMARY_CSV:", SUMMARY_CSV, flush=True)
    print("ALL_TRIAL_CSV:", ALL_TRIAL_CSV, flush=True)
    print("PRED_DIR:", PRED_DIR, flush=True)
    print("TRIAL_DIR:", TRIAL_DIR, flush=True)
    print("PHI_DIR:", PHI_DIR, flush=True)
    print("DIAG_DIR:", DIAG_DIR, flush=True)

## seed runs

In [ ]:
def run_once_fixed_k(run_seed: int):
    seed_everything(run_seed)

    df_full = pd.read_csv(DATA_DIR / FULL_FILE)[["x", "y", "t", "z"]]

    df_run, keep_sites_run, n_sites_full_run = build_fixed_location_subset(
        df=df_full,
        keep_ratio=SPACE_RATIO_KEEP,
        seed=run_seed + 11111,
    )

    df_run["t_norm"] = (
        (df_run["t"] - df_run["t"].min())
        / (df_run["t"].max() - df_run["t"].min() + 1e-12)
    ).astype(np.float32)

    (
        train_mask_flat_run,
        val_mask_flat_run,
        test_mask_flat_run,
        train_time_idx_run,
        val_time_idx_run,
        test_time_idx_run,
        uniq_t_run,
        n_times_run,
    ) = build_contiguous_time_splits(
        df=df_run,
        train_ratio=TRAIN_RATIO_TIME,
        val_ratio=VAL_RATIO_TIME,
        test_ratio=TEST_RATIO_TIME,
    )

    coords_all_run = df_run[["x", "y"]].to_numpy(np.float32)
    t_all_run = df_run["t_norm"].to_numpy(np.float32).reshape(-1, 1)
    y_all_run = df_run["z"].to_numpy(np.float32).reshape(-1, 1)
    X_all_run = np.empty((df_run.shape[0], 0), dtype=np.float32)

    X_train_run = X_all_run[train_mask_flat_run]
    coords_train_run = coords_all_run[train_mask_flat_run]
    t_train_run = t_all_run[train_mask_flat_run]
    y_train_run = y_all_run[train_mask_flat_run]

    train_dataset_run = DictDataset(
        torch.from_numpy(X_train_run),
        torch.from_numpy(coords_train_run),
        torch.from_numpy(t_train_run),
        torch.from_numpy(y_train_run),
    )

    g = torch.Generator()
    g.manual_seed(run_seed + 1000)

    train_loader_run = DataLoader(
        train_dataset_run,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=g,
        num_workers=0,
        pin_memory=True,
        collate_fn=collate_fn,
    )

    stdk_config = build_stdk_model_config(
        EPOCHS=EPOCHS,
        LR=LR,
        WEIGHT_DECAY=WEIGHT_DECAY,
        BATCH_SIZE=BATCH_SIZE,
    )

    stdk_run = create_model(
        stdk_config,
        train_coords=coords_train_run,
    ).to(device)

    stdk_run = train_simple_loop(
        model=stdk_run,
        train_loader=train_loader_run,
        device=device,
        config=stdk_config,
    )

    y_hat_all_run = predict_all_simple(
        model=stdk_run,
        X=torch.from_numpy(X_all_run),
        coords=torch.from_numpy(coords_all_run),
        t=torch.from_numpy(t_all_run),
        batch_size=BATCH_SIZE,
        device=device,
    )

    locs_run, inv_loc_run = np.unique(coords_all_run, axis=0, return_inverse=True)
    t_to_idx_run = {t: i for i, t in enumerate(uniq_t_run)}

    T_run = len(uniq_t_run)
    N_run = len(locs_run)

    t_idx_run = np.array([t_to_idx_run[t] for t in df_run["t"].to_numpy()])
    s_idx_run = inv_loc_run

    y_stdk_run = np.full((T_run, N_run), np.nan, np.float32)
    y_true_run = np.full((T_run, N_run), np.nan, np.float32)

    y_stdk_run[t_idx_run, s_idx_run] = y_hat_all_run
    y_true_run[t_idx_run, s_idx_run] = df_run["z"].to_numpy(np.float32)

    residual_true_run = y_true_run - y_stdk_run

    time_feat_run = (
        (uniq_t_run - uniq_t_run.min())
        / (uniq_t_run.max() - uniq_t_run.min() + 1e-12)
    ).astype(np.float32)

    cont_all_run = (
        torch.from_numpy(time_feat_run)
        .float()
        .unsqueeze(1)
        .repeat(1, N_run)
        .unsqueeze(-1)
    )

    train_cont_train_run = cont_all_run[train_time_idx_run]
    train_y_train_run = torch.from_numpy(
        residual_true_run[train_time_idx_run, :]
    ).float()
    train_idx_train_run = torch.arange(len(train_time_idx_run), dtype=torch.long)

    gna_loader_train_run = DataLoader(
        TensorDataset(train_idx_train_run, train_cont_train_run, train_y_train_run),
        batch_size=min(GNA_BATCH_SIZE, len(train_time_idx_run)),
        shuffle=True,
        drop_last=False,
    )

    residual_true_tensor_all_run = torch.from_numpy(residual_true_run).float()

    seed_phi_dir = PHI_DIR / f"seed_{run_seed}"
    seed_phi_dir.mkdir(parents=True, exist_ok=True)

    semivar_bin_edges = build_standard_bin_edges(locs_run)

    writer_unreg = SummaryWriter(
        str(REPEAT_DIR / "logs_unreg" / f"seed_{run_seed}" / "unreg_tau1_0_tau2_0")
    )
    trend_unreg, basis_unreg = new_trend_basis(N_run, K_FIXED)
    fit_unreg = fit_adapter_reconstruct_all_times(
        tag="unreg_tau1_0_tau2_0",
        tau1=0.0,
        tau2=0.0,
        trend=trend_unreg,
        basis=basis_unreg,
        train_loader=gna_loader_train_run,
        val_cont=train_cont_train_run,
        val_y=train_y_train_run,
        locs=locs_run,
        config=config,
        device=device,
        cont_all=cont_all_run,
        residual_true_tensor_all=residual_true_tensor_all_run,
        writer=writer_unreg,
    )
    writer_unreg.close()

    residual_full_unreg_run = fit_unreg["pred_all"]
    diag_train_unreg_run = fit_unreg["diag_train"]
    diag_all_unreg_run = fit_unreg["diag_all"]
    phi_unreg_run = fit_unreg["phi"]

    np.savez_compressed(
        seed_phi_dir / "unreg_phi.npz",
        phi=phi_unreg_run.astype(np.float32),
        tau1=np.array([0.0], dtype=np.float64),
        tau2=np.array([0.0], dtype=np.float64),
        seed=np.array([run_seed], dtype=np.int32),
    )

    y_final_unreg_run = y_stdk_run + residual_full_unreg_run
    val_rmse_unreg_run = rmse_on_time_subset(
        y_stdk=y_stdk_run,
        y_true=y_true_run,
        pred_all=residual_full_unreg_run,
        time_idx=val_time_idx_run,
    )

    covfrob_stdk_train_base = cov_frob_from_field(
        y_stdk_run[train_time_idx_run, :],
        locs_run,
    )
    covfrob_unreg_train_base = cov_frob_from_field(
        y_final_unreg_run[train_time_idx_run, :],
        locs_run,
    )

    covfrob_stdk_val_base = cov_frob_from_field(
        y_stdk_run[val_time_idx_run, :],
        locs_run,
    )
    covfrob_unreg_val_base = cov_frob_from_field(
        y_final_unreg_run[val_time_idx_run, :],
        locs_run,
    )

    covfrob_stdk_test_base = cov_frob_from_field(
        y_stdk_run[test_time_idx_run, :],
        locs_run,
    )
    covfrob_unreg_test_base = cov_frob_from_field(
        y_final_unreg_run[test_time_idx_run, :],
        locs_run,
    )

    sv_loss_stdk_train_base = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_stdk_run,
            time_idx=train_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_unreg_train_base = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_unreg_run,
            time_idx=train_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )

    sv_loss_stdk_val_base = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_stdk_run,
            time_idx=val_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_unreg_val_base = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_unreg_run,
            time_idx=val_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )

    sv_loss_stdk_test_base = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_stdk_run,
            time_idx=test_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_unreg_test_base = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_unreg_run,
            time_idx=test_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )

    trial_cache = {}
    trial_rows = []

    def objective_run(trial: optuna.Trial):
        tau1 = trial.suggest_float("tau1", TAU_MIN, TAU_MAX, log=True)
        tau2 = trial.suggest_float("tau2", TAU_MIN, TAU_MAX, log=True)

        writer_reg = SummaryWriter(
            str(
                REPEAT_DIR
                / "logs_reg"
                / f"seed_{run_seed}"
                / f"reg_trial_{trial.number:03d}_tau1_{tau1:.2e}_tau2_{tau2:.2e}"
            )
        )

        trend_trial, basis_trial = new_trend_basis(N_run, K_FIXED)
        fit_reg = fit_adapter_reconstruct_all_times(
            tag=f"reg_trial_{trial.number:03d}_tau1_{tau1:.2e}_tau2_{tau2:.2e}",
            tau1=tau1,
            tau2=tau2,
            trend=trend_trial,
            basis=basis_trial,
            train_loader=gna_loader_train_run,
            val_cont=train_cont_train_run,
            val_y=train_y_train_run,
            locs=locs_run,
            config=config,
            device=device,
            cont_all=cont_all_run,
            residual_true_tensor_all=residual_true_tensor_all_run,
            writer=writer_reg,
        )
        writer_reg.close()

        pred_all = fit_reg["pred_all"]
        diag_train = fit_reg["diag_train"]
        diag_all = fit_reg["diag_all"]
        phi_trial = fit_reg["phi"]

        train_rmse = rmse_on_time_subset(
            y_stdk=y_stdk_run,
            y_true=y_true_run,
            pred_all=pred_all,
            time_idx=train_time_idx_run,
        )
        val_rmse = rmse_on_time_subset(
            y_stdk=y_stdk_run,
            y_true=y_true_run,
            pred_all=pred_all,
            time_idx=val_time_idx_run,
        )

        y_final_reg = y_stdk_run + pred_all

        covfrob_reg_train = cov_frob_from_field(
            y_final_reg[train_time_idx_run, :],
            locs_run,
        )
        covfrob_reg_val = cov_frob_from_field(
            y_final_reg[val_time_idx_run, :],
            locs_run,
        )
        covfrob_reg_test = cov_frob_from_field(
            y_final_reg[test_time_idx_run, :],
            locs_run,
        )

        sv_loss_train = float(
            sv_match_loss_for_prediction_matrix(
                coords=locs_run,
                y_true=y_true_run,
                y_pred=y_final_reg,
                time_idx=train_time_idx_run,
                bin_edges=semivar_bin_edges,
                estimator=SEMIVAR_ESTIMATOR,
                weighted=SEMIVAR_WEIGHTED,
                normalized=SEMIVAR_NORMALIZED,
            )["loss"]
        )
        sv_loss_val = float(
            sv_match_loss_for_prediction_matrix(
                coords=locs_run,
                y_true=y_true_run,
                y_pred=y_final_reg,
                time_idx=val_time_idx_run,
                bin_edges=semivar_bin_edges,
                estimator=SEMIVAR_ESTIMATOR,
                weighted=SEMIVAR_WEIGHTED,
                normalized=SEMIVAR_NORMALIZED,
            )["loss"]
        )
        sv_loss_test = float(
            sv_match_loss_for_prediction_matrix(
                coords=locs_run,
                y_true=y_true_run,
                y_pred=y_final_reg,
                time_idx=test_time_idx_run,
                bin_edges=semivar_bin_edges,
                estimator=SEMIVAR_ESTIMATOR,
                weighted=SEMIVAR_WEIGHTED,
                normalized=SEMIVAR_NORMALIZED,
            )["loss"]
        )

        objective_value = choose_objective_value(
            val_rmse=val_rmse,
            covfrob_reg_val=covfrob_reg_val,
            sv_loss_val=sv_loss_val,
        )

        phi_path = seed_phi_dir / f"trial_{trial.number:03d}_phi.npz"
        np.savez_compressed(
            phi_path,
            phi=phi_trial.astype(np.float32),
            tau1=np.array([tau1], dtype=np.float64),
            tau2=np.array([tau2], dtype=np.float64),
            seed=np.array([run_seed], dtype=np.int32),
            trial=np.array([trial.number], dtype=np.int32),
        )

        trial_cache[trial.number] = {
            "tau1": float(tau1),
            "tau2": float(tau2),
            "pred_all": pred_all,
            "diag_train": diag_train,
            "diag_all": diag_all,
            "train_rmse": float(train_rmse),
            "val_rmse": float(val_rmse),
            "sv_loss_train": float(sv_loss_train),
            "sv_loss_val": float(sv_loss_val),
            "sv_loss_test": float(sv_loss_test),
            "covfrob_reg_train": float(covfrob_reg_train),
            "covfrob_reg_val": float(covfrob_reg_val),
            "covfrob_reg_test": float(covfrob_reg_test),
            "phi_path": str(phi_path),
            "objective_value": float(objective_value),
        }

        trial_rows.append({
            "seed": int(run_seed),
            "trial": int(trial.number),
            "tau1": float(tau1),
            "tau2": float(tau2),
            "log10_tau1": float(np.log10(tau1)),
            "log10_tau2": float(np.log10(tau2)),
            "train_rmse": float(train_rmse),
            "val_rmse": float(val_rmse),
            "sv_loss_train": float(sv_loss_train),
            "sv_loss_val": float(sv_loss_val),
            "sv_loss_test": float(sv_loss_test),
            "objective_value": float(objective_value),
            "covfrob_stdk_train": float(covfrob_stdk_train_base),
            "covfrob_unreg_train": float(covfrob_unreg_train_base),
            "covfrob_reg_train": float(covfrob_reg_train),
            "covfrob_stdk_val": float(covfrob_stdk_val_base),
            "covfrob_unreg_val": float(covfrob_unreg_val_base),
            "covfrob_reg_val": float(covfrob_reg_val),
            "covfrob_stdk_test": float(covfrob_stdk_test_base),
            "covfrob_unreg_test": float(covfrob_unreg_test_base),
            "covfrob_reg_test": float(covfrob_reg_test),
            "cov_gain_reg_train": float(covfrob_stdk_train_base - covfrob_reg_train),
            "cov_gain_reg_val": float(covfrob_stdk_val_base - covfrob_reg_val),
            "cov_gain_reg_test": float(covfrob_stdk_test_base - covfrob_reg_test),
            "recon_mse_train": float(diag_train["recon_mse"]),
            "smooth_penalty_train": float(diag_train["smooth_penalty"]),
            "l1_penalty_train": float(diag_train["l1_penalty"]),
            "total_surrogate_train": float(diag_train["total_surrogate"]),
            "n_locations": int(diag_train["n_locations"]),
            "k_basis": int(diag_train["k_basis"]),
            "smooth_penalty_per_entry_train": float(diag_train["smooth_penalty_per_entry"]),
            "l1_penalty_per_entry_train": float(diag_train["l1_penalty_per_entry"]),
            "smooth_over_recon_train": float(diag_train["smooth_over_recon"]),
            "l1_over_recon_train": float(diag_train["l1_over_recon"]),
            "recon_mse_all": float(diag_all["recon_mse"]),
            "smooth_penalty_all": float(diag_all["smooth_penalty"]),
            "l1_penalty_all": float(diag_all["l1_penalty"]),
            "total_surrogate_all": float(diag_all["total_surrogate"]),
            "smooth_penalty_per_entry_all": float(diag_all["smooth_penalty_per_entry"]),
            "l1_penalty_per_entry_all": float(diag_all["l1_penalty_per_entry"]),
            "smooth_over_recon_all": float(diag_all["smooth_over_recon"]),
            "l1_over_recon_all": float(diag_all["l1_over_recon"]),
            "phi_path": str(phi_path),
        })

        print(
            f"[seed {run_seed}] trial {trial.number + 1:03d}/{N_TRIALS:03d} | "
            f"tau1={tau1:.3e} | tau2={tau2:.3e} | "
            f"train_rmse={train_rmse:.6f} | val_rmse={val_rmse:.6f}",
            flush=True,
        )

        return objective_value

    study_run = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=run_seed + 2026),
    )
    study_run.optimize(objective_run, n_trials=N_TRIALS, n_jobs=1)

    trial_df = pd.DataFrame(trial_rows).sort_values("trial").reset_index(drop=True)
    trial_csv_path = TRIAL_DIR / f"seed_{run_seed}_trials.csv"
    trial_df.to_csv(trial_csv_path, index=False)

    if "log10_tau1" not in trial_df.columns:
        trial_df["log10_tau1"] = np.log10(trial_df["tau1"])
    if "log10_tau2" not in trial_df.columns:
        trial_df["log10_tau2"] = np.log10(trial_df["tau2"])

    best_run = study_run.best_trial
    tau1_best_run = float(best_run.params["tau1"])
    tau2_best_run = float(best_run.params["tau2"])
    best_objective_value_run = float(best_run.value)

    residual_full_reg_best_run = trial_cache[best_run.number]["pred_all"]
    diag_train_reg_best_run = trial_cache[best_run.number]["diag_train"]
    diag_all_reg_best_run = trial_cache[best_run.number]["diag_all"]

    phi_reg_best_run = np.load(
        trial_cache[best_run.number]["phi_path"]
    )["phi"].astype(np.float32)

    np.savez_compressed(
        seed_phi_dir / "reg_best_phi.npz",
        phi=phi_reg_best_run.astype(np.float32),
        tau1=np.array([tau1_best_run], dtype=np.float64),
        tau2=np.array([tau2_best_run], dtype=np.float64),
        seed=np.array([run_seed], dtype=np.int32),
        trial=np.array([best_run.number], dtype=np.int32),
    )

    print(
        f"[seed {run_seed}] best trial = {best_run.number + 1:03d} | "
        f"target={TUNING_TARGET} | "
        f"tau1={tau1_best_run:.3e} | tau2={tau2_best_run:.3e} | "
        f"{objective_label()}={best_objective_value_run:.6f}",
        flush=True,
    )

    y_final_reg_best_run = y_stdk_run + residual_full_reg_best_run

    train_mask_run = np.zeros((T_run, N_run), dtype=bool)
    train_mask_run[train_time_idx_run, :] = True

    val_mask_run = np.zeros((T_run, N_run), dtype=bool)
    val_mask_run[val_time_idx_run, :] = True

    test_mask_run = np.zeros((T_run, N_run), dtype=bool)
    test_mask_run[test_time_idx_run, :] = True

    full_mask_run = np.ones((T_run, N_run), dtype=bool)

    stdk_full_run = rmse_pooled(y_true_run, y_stdk_run, full_mask_run)
    unreg_full_run = rmse_pooled(y_true_run, y_final_unreg_run, full_mask_run)
    reg_best_full_run = rmse_pooled(y_true_run, y_final_reg_best_run, full_mask_run)

    stdk_train_run = rmse_pooled(y_true_run, y_stdk_run, train_mask_run)
    unreg_train_run = rmse_pooled(y_true_run, y_final_unreg_run, train_mask_run)
    reg_best_train_run = rmse_pooled(y_true_run, y_final_reg_best_run, train_mask_run)

    stdk_val_run = rmse_pooled(y_true_run, y_stdk_run, val_mask_run)
    unreg_val_run = rmse_pooled(y_true_run, y_final_unreg_run, val_mask_run)
    reg_best_val_run = rmse_pooled(y_true_run, y_final_reg_best_run, val_mask_run)

    stdk_test_run = rmse_pooled(y_true_run, y_stdk_run, test_mask_run)
    unreg_test_run = rmse_pooled(y_true_run, y_final_unreg_run, test_mask_run)
    reg_best_test_run = rmse_pooled(y_true_run, y_final_reg_best_run, test_mask_run)

    covfrob_stdk_full_run = cov_frob_from_field(y_stdk_run, locs_run)
    covfrob_unreg_full_run = cov_frob_from_field(y_final_unreg_run, locs_run)
    covfrob_reg_best_full_run = cov_frob_from_field(y_final_reg_best_run, locs_run)

    covfrob_stdk_train_run = cov_frob_from_field(
        y_stdk_run[train_time_idx_run, :],
        locs_run,
    )
    covfrob_unreg_train_run = cov_frob_from_field(
        y_final_unreg_run[train_time_idx_run, :],
        locs_run,
    )
    covfrob_reg_best_train_run = cov_frob_from_field(
        y_final_reg_best_run[train_time_idx_run, :],
        locs_run,
    )

    covfrob_stdk_val_run = cov_frob_from_field(
        y_stdk_run[val_time_idx_run, :],
        locs_run,
    )
    covfrob_unreg_val_run = cov_frob_from_field(
        y_final_unreg_run[val_time_idx_run, :],
        locs_run,
    )
    covfrob_reg_best_val_run = cov_frob_from_field(
        y_final_reg_best_run[val_time_idx_run, :],
        locs_run,
    )

    covfrob_stdk_test_run = cov_frob_from_field(
        y_stdk_run[test_time_idx_run, :],
        locs_run,
    )
    covfrob_unreg_test_run = cov_frob_from_field(
        y_final_unreg_run[test_time_idx_run, :],
        locs_run,
    )
    covfrob_reg_best_test_run = cov_frob_from_field(
        y_final_reg_best_run[test_time_idx_run, :],
        locs_run,
    )

    sv_loss_stdk_train_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_stdk_run,
            time_idx=train_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_unreg_train_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_unreg_run,
            time_idx=train_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_reg_best_train_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_reg_best_run,
            time_idx=train_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )

    sv_loss_stdk_val_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_stdk_run,
            time_idx=val_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_unreg_val_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_unreg_run,
            time_idx=val_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_reg_best_val_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_reg_best_run,
            time_idx=val_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )

    sv_loss_stdk_test_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_stdk_run,
            time_idx=test_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_unreg_test_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_unreg_run,
            time_idx=test_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )
    sv_loss_reg_best_test_run = float(
        sv_match_loss_for_prediction_matrix(
            coords=locs_run,
            y_true=y_true_run,
            y_pred=y_final_reg_best_run,
            time_idx=test_time_idx_run,
            bin_edges=semivar_bin_edges,
            estimator=SEMIVAR_ESTIMATOR,
            weighted=SEMIVAR_WEIGHTED,
            normalized=SEMIVAR_NORMALIZED,
        )["loss"]
    )

    np.savez_compressed(
        PRED_DIR / f"seed_{run_seed}.npz",
        y_true=y_true_run.astype(np.float32),
        y_stdk=y_stdk_run.astype(np.float32),
        y_unreg=y_final_unreg_run.astype(np.float32),
        y_reg_best=y_final_reg_best_run.astype(np.float32),
        phi_unreg=phi_unreg_run.astype(np.float32),
        phi_reg_best=phi_reg_best_run.astype(np.float32),
        train_mask=train_mask_run.astype(bool),
        val_mask=val_mask_run.astype(bool),
        test_mask=test_mask_run.astype(bool),
        full_mask=full_mask_run.astype(bool),
        locs=locs_run.astype(np.float32),
        keep_sites=keep_sites_run.astype(np.int32),
        train_time_idx=train_time_idx_run.astype(np.int32),
        val_time_idx=val_time_idx_run.astype(np.int32),
        test_time_idx=test_time_idx_run.astype(np.int32),
        n_locations=np.array([N_run], dtype=np.int32),
        k_basis=np.array([K_FIXED], dtype=np.int32),
        tau1_best=np.array([tau1_best_run], dtype=np.float64),
        tau2_best=np.array([tau2_best_run], dtype=np.float64),
        best_objective_value=np.array([best_objective_value_run], dtype=np.float64),
        sv_loss_stdk_train=np.array([sv_loss_stdk_train_run], dtype=np.float64),
        sv_loss_unreg_train=np.array([sv_loss_unreg_train_run], dtype=np.float64),
        sv_loss_reg_best_train=np.array([sv_loss_reg_best_train_run], dtype=np.float64),
        sv_loss_stdk_val=np.array([sv_loss_stdk_val_run], dtype=np.float64),
        sv_loss_unreg_val=np.array([sv_loss_unreg_val_run], dtype=np.float64),
        sv_loss_reg_best_val=np.array([sv_loss_reg_best_val_run], dtype=np.float64),
        sv_loss_stdk_test=np.array([sv_loss_stdk_test_run], dtype=np.float64),
        sv_loss_unreg_test=np.array([sv_loss_unreg_test_run], dtype=np.float64),
        sv_loss_reg_best_test=np.array([sv_loss_reg_best_test_run], dtype=np.float64),
        unreg_recon_mse_train=np.array([diag_train_unreg_run["recon_mse"]], dtype=np.float64),
        unreg_smooth_penalty_train=np.array([diag_train_unreg_run["smooth_penalty"]], dtype=np.float64),
        unreg_l1_penalty_train=np.array([diag_train_unreg_run["l1_penalty"]], dtype=np.float64),
        unreg_total_surrogate_train=np.array([diag_train_unreg_run["total_surrogate"]], dtype=np.float64),
        unreg_smooth_penalty_per_entry_train=np.array([diag_train_unreg_run["smooth_penalty_per_entry"]], dtype=np.float64),
        unreg_l1_penalty_per_entry_train=np.array([diag_train_unreg_run["l1_penalty_per_entry"]], dtype=np.float64),
        unreg_smooth_over_recon_train=np.array([diag_train_unreg_run["smooth_over_recon"]], dtype=np.float64),
        unreg_l1_over_recon_train=np.array([diag_train_unreg_run["l1_over_recon"]], dtype=np.float64),
        reg_best_recon_mse_train=np.array([diag_train_reg_best_run["recon_mse"]], dtype=np.float64),
        reg_best_smooth_penalty_train=np.array([diag_train_reg_best_run["smooth_penalty"]], dtype=np.float64),
        reg_best_l1_penalty_train=np.array([diag_train_reg_best_run["l1_penalty"]], dtype=np.float64),
        reg_best_total_surrogate_train=np.array([diag_train_reg_best_run["total_surrogate"]], dtype=np.float64),
        reg_best_smooth_penalty_per_entry_train=np.array([diag_train_reg_best_run["smooth_penalty_per_entry"]], dtype=np.float64),
        reg_best_l1_penalty_per_entry_train=np.array([diag_train_reg_best_run["l1_penalty_per_entry"]], dtype=np.float64),
        reg_best_smooth_over_recon_train=np.array([diag_train_reg_best_run["smooth_over_recon"]], dtype=np.float64),
        reg_best_l1_over_recon_train=np.array([diag_train_reg_best_run["l1_over_recon"]], dtype=np.float64),
        unreg_recon_mse_all=np.array([diag_all_unreg_run["recon_mse"]], dtype=np.float64),
        unreg_smooth_penalty_all=np.array([diag_all_unreg_run["smooth_penalty"]], dtype=np.float64),
        unreg_l1_penalty_all=np.array([diag_all_unreg_run["l1_penalty"]], dtype=np.float64),
        unreg_total_surrogate_all=np.array([diag_all_unreg_run["total_surrogate"]], dtype=np.float64),
        unreg_smooth_penalty_per_entry_all=np.array([diag_all_unreg_run["smooth_penalty_per_entry"]], dtype=np.float64),
        unreg_l1_penalty_per_entry_all=np.array([diag_all_unreg_run["l1_penalty_per_entry"]], dtype=np.float64),
        unreg_smooth_over_recon_all=np.array([diag_all_unreg_run["smooth_over_recon"]], dtype=np.float64),
        unreg_l1_over_recon_all=np.array([diag_all_unreg_run["l1_over_recon"]], dtype=np.float64),
        reg_best_recon_mse_all=np.array([diag_all_reg_best_run["recon_mse"]], dtype=np.float64),
        reg_best_smooth_penalty_all=np.array([diag_all_reg_best_run["smooth_penalty"]], dtype=np.float64),
        reg_best_l1_penalty_all=np.array([diag_all_reg_best_run["l1_penalty"]], dtype=np.float64),
        reg_best_total_surrogate_all=np.array([diag_all_reg_best_run["total_surrogate"]], dtype=np.float64),
        reg_best_smooth_penalty_per_entry_all=np.array([diag_all_reg_best_run["smooth_penalty_per_entry"]], dtype=np.float64),
        reg_best_l1_penalty_per_entry_all=np.array([diag_all_reg_best_run["l1_penalty_per_entry"]], dtype=np.float64),
        reg_best_smooth_over_recon_all=np.array([diag_all_reg_best_run["smooth_over_recon"]], dtype=np.float64),
        reg_best_l1_over_recon_all=np.array([diag_all_reg_best_run["l1_over_recon"]], dtype=np.float64),
    )

    return {
        "seed": int(run_seed),
        "tuning_target": TUNING_TARGET,
        "n_sites_full": int(n_sites_full_run),
        "n_sites_kept": int(len(keep_sites_run)),
        "tau1_best": float(tau1_best_run),
        "tau2_best": float(tau2_best_run),
        "best_objective_value": float(best_objective_value_run),
        "val_rmse_unreg": float(val_rmse_unreg_run),
        "stdk_full": float(stdk_full_run),
        "unreg_full": float(unreg_full_run),
        "reg_best_full": float(reg_best_full_run),
        "stdk_train": float(stdk_train_run),
        "unreg_train": float(unreg_train_run),
        "reg_best_train": float(reg_best_train_run),
        "stdk_val": float(stdk_val_run),
        "unreg_val": float(unreg_val_run),
        "reg_best_val": float(reg_best_val_run),
        "stdk_test": float(stdk_test_run),
        "unreg_test": float(unreg_test_run),
        "reg_best_test": float(reg_best_test_run),
        "covfrob_stdk_full": float(covfrob_stdk_full_run),
        "covfrob_unreg_full": float(covfrob_unreg_full_run),
        "covfrob_reg_best_full": float(covfrob_reg_best_full_run),
        "covfrob_stdk_train": float(covfrob_stdk_train_run),
        "covfrob_unreg_train": float(covfrob_unreg_train_run),
        "covfrob_reg_best_train": float(covfrob_reg_best_train_run),
        "covfrob_stdk_val": float(covfrob_stdk_val_run),
        "covfrob_unreg_val": float(covfrob_unreg_val_run),
        "covfrob_reg_best_val": float(covfrob_reg_best_val_run),
        "covfrob_stdk_test": float(covfrob_stdk_test_run),
        "covfrob_unreg_test": float(covfrob_unreg_test_run),
        "covfrob_reg_best_test": float(covfrob_reg_best_test_run),
        "sv_loss_stdk_train": float(sv_loss_stdk_train_run),
        "sv_loss_unreg_train": float(sv_loss_unreg_train_run),
        "sv_loss_reg_best_train": float(sv_loss_reg_best_train_run),
        "sv_loss_stdk_val": float(sv_loss_stdk_val_run),
        "sv_loss_unreg_val": float(sv_loss_unreg_val_run),
        "sv_loss_reg_best_val": float(sv_loss_reg_best_val_run),
        "sv_loss_stdk_test": float(sv_loss_stdk_test_run),
        "sv_loss_unreg_test": float(sv_loss_unreg_test_run),
        "sv_loss_reg_best_test": float(sv_loss_reg_best_test_run),
        "unreg_recon_mse_train": float(diag_train_unreg_run["recon_mse"]),
        "unreg_smooth_penalty_train": float(diag_train_unreg_run["smooth_penalty"]),
        "unreg_l1_penalty_train": float(diag_train_unreg_run["l1_penalty"]),
        "unreg_total_surrogate_train": float(diag_train_unreg_run["total_surrogate"]),
        "unreg_smooth_penalty_per_entry_train": float(diag_train_unreg_run["smooth_penalty_per_entry"]),
        "unreg_l1_penalty_per_entry_train": float(diag_train_unreg_run["l1_penalty_per_entry"]),
        "unreg_smooth_over_recon_train": float(diag_train_unreg_run["smooth_over_recon"]),
        "unreg_l1_over_recon_train": float(diag_train_unreg_run["l1_over_recon"]),
        "reg_best_recon_mse_train": float(diag_train_reg_best_run["recon_mse"]),
        "reg_best_smooth_penalty_train": float(diag_train_reg_best_run["smooth_penalty"]),
        "reg_best_l1_penalty_train": float(diag_train_reg_best_run["l1_penalty"]),
        "reg_best_total_surrogate_train": float(diag_train_reg_best_run["total_surrogate"]),
        "reg_best_smooth_penalty_per_entry_train": float(diag_train_reg_best_run["smooth_penalty_per_entry"]),
        "reg_best_l1_penalty_per_entry_train": float(diag_train_reg_best_run["l1_penalty_per_entry"]),
        "reg_best_smooth_over_recon_train": float(diag_train_reg_best_run["smooth_over_recon"]),
        "reg_best_l1_over_recon_train": float(diag_train_reg_best_run["l1_over_recon"]),
        "unreg_recon_mse_all": float(diag_all_unreg_run["recon_mse"]),
        "unreg_smooth_penalty_all": float(diag_all_unreg_run["smooth_penalty"]),
        "unreg_l1_penalty_all": float(diag_all_unreg_run["l1_penalty"]),
        "unreg_total_surrogate_all": float(diag_all_unreg_run["total_surrogate"]),
        "unreg_smooth_penalty_per_entry_all": float(diag_all_unreg_run["smooth_penalty_per_entry"]),
        "unreg_l1_penalty_per_entry_all": float(diag_all_unreg_run["l1_penalty_per_entry"]),
        "unreg_smooth_over_recon_all": float(diag_all_unreg_run["smooth_over_recon"]),
        "unreg_l1_over_recon_all": float(diag_all_unreg_run["l1_over_recon"]),
        "reg_best_recon_mse_all": float(diag_all_reg_best_run["recon_mse"]),
        "reg_best_smooth_penalty_all": float(diag_all_reg_best_run["smooth_penalty"]),
        "reg_best_l1_penalty_all": float(diag_all_reg_best_run["l1_penalty"]),
        "reg_best_total_surrogate_all": float(diag_all_reg_best_run["total_surrogate"]),
        "reg_best_smooth_penalty_per_entry_all": float(diag_all_reg_best_run["smooth_penalty_per_entry"]),
        "reg_best_l1_penalty_per_entry_all": float(diag_all_reg_best_run["l1_penalty_per_entry"]),
        "reg_best_smooth_over_recon_all": float(diag_all_reg_best_run["smooth_over_recon"]),
        "reg_best_l1_over_recon_all": float(diag_all_reg_best_run["l1_over_recon"]),
    }

## run

In [ ]:
rows_fixed_k = []

for r in range(N_RUNS_FIXED_K):
    run_seed = SEED + r * 1000
    print(
        f"\n===== FIXED K RUN {r + 1}/{N_RUNS_FIXED_K} | "
        f"seed={run_seed} | target={TUNING_TARGET} | "
        f"space_keep={SPACE_RATIO_KEEP} | "
        f"time_train={TRAIN_RATIO_TIME} | "
        f"time_val={VAL_RATIO_TIME} | "
        f"time_test={TEST_RATIO_TIME} =====",
        flush=True,
    )
    rows_fixed_k.append(run_once_fixed_k(run_seed))

results_fixed_k_df = pd.DataFrame(rows_fixed_k)
results_fixed_k_df.to_csv(SUMMARY_CSV, index=False)

trial_files = sorted(TRIAL_DIR.glob("seed_*_trials.csv"))
if len(trial_files) > 0:
    all_trial_df = pd.concat(
        [pd.read_csv(f) for f in trial_files],
        ignore_index=True,
    )
    all_trial_df.to_csv(ALL_TRIAL_CSV, index=False)
else:
    all_trial_df = pd.DataFrame()

print("\n=== Summary files saved ===", flush=True)
print("SUMMARY_CSV:", SUMMARY_CSV, flush=True)
print("ALL_TRIAL_CSV:", ALL_TRIAL_CSV, flush=True)

## summary

In [ ]:
results_fixed_k_df = load_summary_or_empty()

if results_fixed_k_df.empty:
    print("SUMMARY_CSV not found or empty:", SUMMARY_CSV, flush=True)
else:
    print("\n=== RMSE summary ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(results_fixed_k_df[f'stdk_{split}'])} | "
            f"unreg = {fmt_pm(results_fixed_k_df[f'unreg_{split}'])} | "
            f"reg_best = {fmt_pm(results_fixed_k_df[f'reg_best_{split}'])}",
            flush=True,
        )

    print("\n=== CovFrob summary ===", flush=True)
    for split in ["full", "train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(results_fixed_k_df[f'covfrob_stdk_{split}'])} | "
            f"unreg = {fmt_pm(results_fixed_k_df[f'covfrob_unreg_{split}'])} | "
            f"reg_best = {fmt_pm(results_fixed_k_df[f'covfrob_reg_best_{split}'])}",
            flush=True,
        )

    print("\n=== Semivariogram matching loss summary ===", flush=True)
    for split in ["train", "val", "test"]:
        print(
            f"{split:>5} | "
            f"stdk = {fmt_pm(results_fixed_k_df[f'sv_loss_stdk_{split}'])} | "
            f"unreg = {fmt_pm(results_fixed_k_df[f'sv_loss_unreg_{split}'])} | "
            f"reg_best = {fmt_pm(results_fixed_k_df[f'sv_loss_reg_best_{split}'])}",
            flush=True,
        )

    print("\n=== Selected tau summary ===", flush=True)
    print("tau1_best:", fmt_pm(results_fixed_k_df["tau1_best"]), flush=True)
    print("tau2_best:", fmt_pm(results_fixed_k_df["tau2_best"]), flush=True)
    print(f"{objective_label()}:", fmt_pm(results_fixed_k_df["best_objective_value"]), flush=True)

    print("\n=== Basic run info ===", flush=True)
    print("tuning_target:", TUNING_TARGET, flush=True)
    print("n_runs:", N_RUNS_FIXED_K, flush=True)
    print("space_ratio_keep:", SPACE_RATIO_KEEP, flush=True)
    print(
        f"time split = ({TRAIN_RATIO_TIME}, {VAL_RATIO_TIME}, {TEST_RATIO_TIME})",
        flush=True,
    )

## heat map

In [ ]:
# df_all_trials = load_all_trials_or_empty()

# if df_all_trials.empty:
#     print("ALL_TRIAL_CSV not found or empty:", ALL_TRIAL_CSV, flush=True)
# else:
#     if "log10_tau1" not in df_all_trials.columns:
#         df_all_trials["log10_tau1"] = np.log10(df_all_trials["tau1"])

#     if "log10_tau2" not in df_all_trials.columns:
#         df_all_trials["log10_tau2"] = np.log10(df_all_trials["tau2"])

#     plot_trial_maps(
#         df=df_all_trials.copy(),
#         output_dir=TRIAL_DIR / "plots_all",
#         title_suffix=f"(All seeds, {TUNING_TARGET} tuning)",
#         file_prefix=f"all_seeds_{TUNING_TARGET}",
#         best_by=objective_best_by_column(),
#     )

#     seed_list = sorted(df_all_trials["seed"].dropna().unique().tolist())

#     for seed in seed_list:
#         df_seed = df_all_trials[df_all_trials["seed"] == seed].copy()
#         if df_seed.empty:
#             continue

#         if "log10_tau1" not in df_seed.columns:
#             df_seed["log10_tau1"] = np.log10(df_seed["tau1"])

#         if "log10_tau2" not in df_seed.columns:
#             df_seed["log10_tau2"] = np.log10(df_seed["tau2"])

#         plot_trial_maps(
#             df=df_seed,
#             output_dir=TRIAL_DIR / f"plots_seed_{int(seed)}",
#             title_suffix=f"(seed {int(seed)}, {TUNING_TARGET} tuning)",
#             file_prefix=f"seed_{int(seed)}_{TUNING_TARGET}",
#             best_by=objective_best_by_column(),
#         )

#     print("\n=== Heat map finished ===", flush=True)
#     print("Loaded:", ALL_TRIAL_CSV, flush=True)
#     print("Saved under:", TRIAL_DIR, flush=True)

## diagnostics

In [ ]:
# # --------------------------------------------------
# # A. Local helpers for eigen diagnostics
# # --------------------------------------------------
# def empirical_cov_from_field(Y):
#     Y = np.asarray(Y, dtype=float)
#     if Y.ndim != 2:
#         raise ValueError("Y must be 2D with shape (n_time, n_locations)")
#     if Y.shape[0] < 2:
#         return np.full((Y.shape[1], Y.shape[1]), np.nan, dtype=float)
#     Yc = Y - np.mean(Y, axis=0, keepdims=True)
#     return (Yc.T @ Yc) / (Y.shape[0] - 1)


# def sorted_eigh(S):
#     vals, vecs = np.linalg.eigh(S)
#     order = np.argsort(vals)[::-1]
#     vals = vals[order]
#     vecs = vecs[:, order]
#     return vals, vecs


# def safe_topk(k, n):
#     return int(max(1, min(k, n)))


# def eigenvector_alignment(vec_true, vec_pred):
#     vec_true = np.asarray(vec_true, dtype=float).reshape(-1)
#     vec_pred = np.asarray(vec_pred, dtype=float).reshape(-1)

#     n_true = np.linalg.norm(vec_true)
#     n_pred = np.linalg.norm(vec_pred)
#     if n_true < 1e-12 or n_pred < 1e-12:
#         return np.nan

#     return float(np.abs(np.dot(vec_true, vec_pred) / (n_true * n_pred)))


# def cumulative_variance_ratio(eigvals, topk):
#     eigvals = np.asarray(eigvals, dtype=float)
#     eigvals = np.clip(eigvals, a_min=0.0, a_max=None)
#     denom = np.sum(eigvals)
#     if denom < 1e-12:
#         return np.nan
#     return float(np.sum(eigvals[:topk]) / denom)


# def topk_relative_eig_error(eig_true, eig_pred, topk, eps=1e-12):
#     eig_true = np.asarray(eig_true[:topk], dtype=float)
#     eig_pred = np.asarray(eig_pred[:topk], dtype=float)
#     return float(np.mean(np.abs(eig_pred - eig_true) / np.maximum(np.abs(eig_true), eps)))


# def topk_absolute_eig_error(eig_true, eig_pred, topk):
#     eig_true = np.asarray(eig_true[:topk], dtype=float)
#     eig_pred = np.asarray(eig_pred[:topk], dtype=float)
#     return float(np.mean(np.abs(eig_pred - eig_true)))


# def save_eigenspectrum_plot(eig_true, eig_stdk, eig_unreg, eig_reg, seed, split_name, out_dir, topk=20):
#     topk = safe_topk(topk, len(eig_true))

#     x = np.arange(1, topk + 1)

#     fig = plt.figure(figsize=(6, 4))
#     plt.plot(x, eig_true[:topk], label="true")
#     plt.plot(x, eig_stdk[:topk], label="stdk")
#     plt.plot(x, eig_unreg[:topk], label="unreg")
#     plt.plot(x, eig_reg[:topk], label="reg_best")
#     plt.xlabel("Eigenvalue rank")
#     plt.ylabel("Eigenvalue")
#     plt.title(f"Eigenspectrum | seed={seed} | split={split_name}")
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(out_dir / f"seed_{seed}_{split_name}_eigenspectrum_top{topk}.png", dpi=200)
#     plt.close()


# def save_leading_eigenvector_plot(vec_true, vec_stdk, vec_unreg, vec_reg, seed, split_name, out_dir):
#     fig = plt.figure(figsize=(8, 4))
#     plt.plot(vec_true, label="true")
#     plt.plot(vec_stdk, label="stdk")
#     plt.plot(vec_unreg, label="unreg")
#     plt.plot(vec_reg, label="reg_best")
#     plt.xlabel("Location index")
#     plt.ylabel("Eigenvector value")
#     plt.title(f"Leading eigenvector | seed={seed} | split={split_name}")
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(out_dir / f"seed_{seed}_{split_name}_leading_eigenvector.png", dpi=200)
#     plt.close()


# # --------------------------------------------------
# # B. Load summary and saved predictions
# # --------------------------------------------------
# summary_df = load_summary_or_empty()
# npz_files = list_prediction_npz_files()

# if len(npz_files) == 0:
#     print("No saved prediction files found in:", PRED_DIR, flush=True)

# eigen_dir = DIAG_DIR / "eigen"
# eigen_dir.mkdir(parents=True, exist_ok=True)

# eigen_rows = []

# # --------------------------------------------------
# # C. Compute eigen diagnostics from saved predictions
# # --------------------------------------------------
# for npz_path in npz_files:
#     with np.load(npz_path, allow_pickle=True) as data:
#         seed_str = npz_path.stem.replace("seed_", "")
#         seed = int(seed_str) if seed_str.isdigit() else np.nan

#         y_true = data["y_true"]
#         y_stdk = data["y_stdk"]
#         y_unreg = data["y_unreg"]
#         y_reg_best = data["y_reg_best"]

#         train_time_idx = data["train_time_idx"]
#         val_time_idx = data["val_time_idx"]
#         test_time_idx = data["test_time_idx"]

#         split_dict = {
#             "train": train_time_idx,
#             "val": val_time_idx,
#             "test": test_time_idx,
#         }

#         for split_name, time_idx in split_dict.items():
#             Y_true_split = y_true[time_idx, :]
#             Y_stdk_split = y_stdk[time_idx, :]
#             Y_unreg_split = y_unreg[time_idx, :]
#             Y_reg_split = y_reg_best[time_idx, :]

#             S_true = empirical_cov_from_field(Y_true_split)
#             S_stdk = empirical_cov_from_field(Y_stdk_split)
#             S_unreg = empirical_cov_from_field(Y_unreg_split)
#             S_reg = empirical_cov_from_field(Y_reg_split)

#             if not np.all(np.isfinite(S_true)):
#                 continue
#             if not np.all(np.isfinite(S_stdk)):
#                 continue
#             if not np.all(np.isfinite(S_unreg)):
#                 continue
#             if not np.all(np.isfinite(S_reg)):
#                 continue

#             eig_true, vec_true = sorted_eigh(S_true)
#             eig_stdk, vec_stdk = sorted_eigh(S_stdk)
#             eig_unreg, vec_unreg = sorted_eigh(S_unreg)
#             eig_reg, vec_reg = sorted_eigh(S_reg)

#             top1 = safe_topk(1, len(eig_true))
#             top3 = safe_topk(3, len(eig_true))
#             top5 = safe_topk(5, len(eig_true))
#             top10 = safe_topk(10, len(eig_true))

#             save_eigenspectrum_plot(
#                 eig_true=eig_true,
#                 eig_stdk=eig_stdk,
#                 eig_unreg=eig_unreg,
#                 eig_reg=eig_reg,
#                 seed=seed,
#                 split_name=split_name,
#                 out_dir=eigen_dir,
#                 topk=20,
#             )

#             save_leading_eigenvector_plot(
#                 vec_true=vec_true[:, 0],
#                 vec_stdk=vec_stdk[:, 0],
#                 vec_unreg=vec_unreg[:, 0],
#                 vec_reg=vec_reg[:, 0],
#                 seed=seed,
#                 split_name=split_name,
#                 out_dir=eigen_dir,
#             )

#             eigen_rows.append({
#                 "seed": seed,
#                 "split": split_name,

#                 "cumvar_true_top1": cumulative_variance_ratio(eig_true, top1),
#                 "cumvar_stdk_top1": cumulative_variance_ratio(eig_stdk, top1),
#                 "cumvar_unreg_top1": cumulative_variance_ratio(eig_unreg, top1),
#                 "cumvar_reg_best_top1": cumulative_variance_ratio(eig_reg, top1),

#                 "cumvar_true_top3": cumulative_variance_ratio(eig_true, top3),
#                 "cumvar_stdk_top3": cumulative_variance_ratio(eig_stdk, top3),
#                 "cumvar_unreg_top3": cumulative_variance_ratio(eig_unreg, top3),
#                 "cumvar_reg_best_top3": cumulative_variance_ratio(eig_reg, top3),

#                 "cumvar_true_top5": cumulative_variance_ratio(eig_true, top5),
#                 "cumvar_stdk_top5": cumulative_variance_ratio(eig_stdk, top5),
#                 "cumvar_unreg_top5": cumulative_variance_ratio(eig_unreg, top5),
#                 "cumvar_reg_best_top5": cumulative_variance_ratio(eig_reg, top5),

#                 "cumvar_true_top10": cumulative_variance_ratio(eig_true, top10),
#                 "cumvar_stdk_top10": cumulative_variance_ratio(eig_stdk, top10),
#                 "cumvar_unreg_top10": cumulative_variance_ratio(eig_unreg, top10),
#                 "cumvar_reg_best_top10": cumulative_variance_ratio(eig_reg, top10),

#                 "eig_abs_err_stdk_top3": topk_absolute_eig_error(eig_true, eig_stdk, top3),
#                 "eig_abs_err_unreg_top3": topk_absolute_eig_error(eig_true, eig_unreg, top3),
#                 "eig_abs_err_reg_best_top3": topk_absolute_eig_error(eig_true, eig_reg, top3),

#                 "eig_rel_err_stdk_top3": topk_relative_eig_error(eig_true, eig_stdk, top3),
#                 "eig_rel_err_unreg_top3": topk_relative_eig_error(eig_true, eig_unreg, top3),
#                 "eig_rel_err_reg_best_top3": topk_relative_eig_error(eig_true, eig_reg, top3),

#                 "eigvec_align_stdk_top1": eigenvector_alignment(vec_true[:, 0], vec_stdk[:, 0]),
#                 "eigvec_align_unreg_top1": eigenvector_alignment(vec_true[:, 0], vec_unreg[:, 0]),
#                 "eigvec_align_reg_best_top1": eigenvector_alignment(vec_true[:, 0], vec_reg[:, 0]),

#                 "eigvec_align_stdk_top2": eigenvector_alignment(vec_true[:, 1], vec_stdk[:, 1]) if vec_true.shape[1] > 1 else np.nan,
#                 "eigvec_align_unreg_top2": eigenvector_alignment(vec_true[:, 1], vec_unreg[:, 1]) if vec_true.shape[1] > 1 else np.nan,
#                 "eigvec_align_reg_best_top2": eigenvector_alignment(vec_true[:, 1], vec_reg[:, 1]) if vec_true.shape[1] > 1 else np.nan,

#                 "eigvec_align_stdk_top3": eigenvector_alignment(vec_true[:, 2], vec_stdk[:, 2]) if vec_true.shape[1] > 2 else np.nan,
#                 "eigvec_align_unreg_top3": eigenvector_alignment(vec_true[:, 2], vec_unreg[:, 2]) if vec_true.shape[1] > 2 else np.nan,
#                 "eigvec_align_reg_best_top3": eigenvector_alignment(vec_true[:, 2], vec_reg[:, 2]) if vec_true.shape[1] > 2 else np.nan,
#             })

# # --------------------------------------------------
# # D. Save eigen diagnostics table
# # --------------------------------------------------
# eigen_df = pd.DataFrame(eigen_rows)

# if eigen_df.empty:
#     print("No eigen diagnostics rows were created.", flush=True)
# else:
#     EIGEN_CSV = DIAG_DIR / "eigen_diagnostics.csv"
#     eigen_df.to_csv(EIGEN_CSV, index=False)

#     print("\n=== Eigen diagnostics saved ===", flush=True)
#     print("EIGEN_CSV:", EIGEN_CSV, flush=True)
#     print("EIGEN_PLOT_DIR:", eigen_dir, flush=True)

#     print("\n=== Eigen alignment summary ===", flush=True)
#     for split_name in ["train", "val", "test"]:
#         df_split = eigen_df[eigen_df["split"] == split_name]
#         if df_split.empty:
#             continue
#         print(
#             f"{split_name:>5} | "
#             f"top1 align stdk = {fmt_pm(df_split['eigvec_align_stdk_top1'])} | "
#             f"unreg = {fmt_pm(df_split['eigvec_align_unreg_top1'])} | "
#             f"reg_best = {fmt_pm(df_split['eigvec_align_reg_best_top1'])}",
#             flush=True,
#         )

#     print("\n=== Eigenvalue relative error summary (top3) ===", flush=True)
#     for split_name in ["train", "val", "test"]:
#         df_split = eigen_df[eigen_df["split"] == split_name]
#         if df_split.empty:
#             continue
#         print(
#             f"{split_name:>5} | "
#             f"stdk = {fmt_pm(df_split['eig_rel_err_stdk_top3'])} | "
#             f"unreg = {fmt_pm(df_split['eig_rel_err_unreg_top3'])} | "
#             f"reg_best = {fmt_pm(df_split['eig_rel_err_reg_best_top3'])}",
#             flush=True,
#         )

# # --------------------------------------------------
# # E. Optional summary consistency check
# #    This checks whether the summary file exists and can
# #    be joined with the eigen diagnostics by seed
# # --------------------------------------------------
# if not summary_df.empty and not eigen_df.empty and "seed" in summary_df.columns and "seed" in eigen_df.columns:
#     eigen_seed_counts = (
#         eigen_df.groupby("seed")
#         .size()
#         .reset_index(name="n_eigen_rows")
#     )

#     summary_eigen_check = summary_df.merge(
#         eigen_seed_counts,
#         on="seed",
#         how="left",
#     )

#     SUMMARY_EIGEN_CHECK_CSV = DIAG_DIR / "summary_eigen_check.csv"
#     summary_eigen_check.to_csv(SUMMARY_EIGEN_CHECK_CSV, index=False)

#     print("\n=== Summary-eigen check saved ===", flush=True)
#     print("SUMMARY_EIGEN_CHECK_CSV:", SUMMARY_EIGEN_CHECK_CSV, flush=True)